# Projeto AM-NS 2026-2 - Equipe 8
Agrupamento e redução de dimensionalidade na base Iris.

Algoritmos da Equipe 8: K-means, Fuzzy K-means, Kernel Fuzzy K-means, K-medoids (euclidiana), Misturas EM, UPGMA (city-block) e DBSCAN.
Questão 2: PCA, Kernel PCA, t-SNE e UMAP.

## Preparação

Em Jupyter, abra o notebook a partir da cópia local do projeto com o ambiente Python do README.
No Colab, se a pasta do projeto estiver ausente, a primeira célula solicita um ZIP com a pasta
`codigo/` da mesma versão do notebook. O ZIP deve conter os scripts Python e `codigo/iris/iris.data`.

In [ ]:
import io
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

arquivos = (
    "algoritmos.py", "questoes.py", "tunagem.py", "iris/iris.data",
)
candidatos = (Path.cwd() / "codigo", Path.cwd())
pasta_codigo = next(
    (pasta for pasta in candidatos if all((pasta / nome).is_file() for nome in arquivos)),
    None,
)

no_colab = "google.colab" in sys.modules
if pasta_codigo is None:
    if not no_colab:
        raise FileNotFoundError("Abra o notebook a partir da raiz do projeto ou da pasta codigo.")
    from google.colab import files

    uploads = files.upload()
    if len(uploads) != 1:
        raise ValueError("Selecione um ZIP com a pasta codigo do projeto.")
    pasta_upload = TemporaryDirectory(prefix="projeto-amns-")
    pasta_codigo = Path(pasta_upload.name) / "codigo"
    with ZipFile(io.BytesIO(next(iter(uploads.values())))) as zip_projeto:
        nomes = zip_projeto.namelist()
        for nome in arquivos:
            sufixo = f"codigo/{nome}"
            membros = [m for m in nomes if m == sufixo or m.endswith(f"/{sufixo}")]
            if len(membros) != 1:
                raise ValueError(f"O ZIP precisa conter uma cópia de {sufixo}.")
            destino = pasta_codigo / nome
            destino.parent.mkdir(parents=True, exist_ok=True)
            destino.write_bytes(zip_projeto.read(membros[0]))

if no_colab:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
         "fuzzy-c-means==2.3.1", "kmedoids==0.5.5"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=True)
    sys.modules["tensorflow"] = None

sys.path.insert(0, str(pasta_codigo.resolve()))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import TSNE, trustworthiness
from umap import UMAP

from algoritmos import carregar_iris
from questoes import executar_questao_1

X, y, pre = carregar_iris()
ATRIBUTOS = pre["atributos"]
GAMMA_KERNEL = 0.25
PERPLEXIDADE, VIZINHOS_UMAP, MIN_DIST_UMAP, SEED_Q2 = 30, 15, 0.1, 42
sns.set_theme(style="whitegrid")

## Questão 1 — Agrupamento

A execução usa os mesmos scripts do projeto. Para cada configuração particional e cada
`k=2..5`, a função objetivo escolhe uma das dez execuções. A silhueta escolhe a configuração
e `k*`. A silhueta em `k=1` é indefinida. O UPGMA usa ligação média e Manhattan.

O DBSCAN seleciona pela silhueta com cobertura mínima de 95%. Sua silhueta e seu ARI
são calculados após remover o ruído. O ARI avalia a concordância com as espécies depois da escolha.

In [ ]:
pasta_saida = TemporaryDirectory(prefix="amns-q1-")
resultados_q1 = Path(pasta_saida.name)
resumo = executar_questao_1(diretorio_resultados=resultados_q1)
display(resumo.round(4))

### Número de grupos, dendrograma e comparação na mesma amostra

In [ ]:
metricas = pd.read_csv(resultados_q1 / "metricas_por_k.csv")
for metrica in ("silhueta", "ari"):
    display(metricas.pivot(index="algoritmo", columns="k", values=metrica).round(4))
for figura in sorted((resultados_q1 / "figuras").glob("*.png")):
    display(Image(filename=str(figura), width=650))
comparacao = pd.read_csv(resultados_q1 / "tunagem" / "comparacao_mesma_amostra.csv")
display(comparacao.round(4))

### Sensibilidade aos hiperparâmetros

As tabelas mostram as configurações avaliadas. Na leitura do DBSCAN, acompanhe a
silhueta e o ARI com a cobertura. Nas curvas particionais, compare cada configuração
ao longo dos valores de `k`.

In [ ]:
tunagem = resultados_q1 / "tunagem"
for tabela in ("particionais", "dbscan"):
    dados = pd.read_csv(tunagem / f"{tabela}.csv")
    display(dados.round(4))
for figura in sorted((tunagem / "figuras").glob("*.png")):
    display(Image(filename=str(figura), width=750))

## Questão 2 — PCA, Kernel PCA, t-SNE e UMAP
Todas as técnicas usam a mesma matriz padronizada com os 4 atributos. Para cada uma mostramos as saídas, o gráfico 2D e a trustworthiness (quanto dos 10 vizinhos mais próximos foi preservado, de 0 a 1).

In [ ]:
def scatter2d(Z, titulo, xl="Componente 1", yl="Componente 2"):
    fig, ax = plt.subplots(figsize=(8, 5.5))
    sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=y, style=y, s=55, ax=ax)
    ax.set(title=titulo, xlabel=xl, ylabel=yl)
    ax.legend(title="Espécie", fontsize=8)
    imagem = io.BytesIO()
    fig.savefig(imagem, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    display(Image(data=imagem.getvalue()))

trust = {}

### 1 PCA

In [ ]:
pca = PCA(n_components=4, svd_solver="full")
Z_pca = pca.fit_transform(X)
cp = [f"PC{i+1}" for i in range(4)]

saida_pca = pd.DataFrame({"Autovalor": pca.explained_variance_,
                          "Variância (%)": 100 * pca.explained_variance_ratio_,
                          "Acumulada (%)": 100 * np.cumsum(pca.explained_variance_ratio_),
                          "Valor singular": pca.singular_values_}, index=cp)
display(saida_pca.round(4))

print("Componentes (autovetores):")
display(pd.DataFrame(pca.components_.T, index=ATRIBUTOS, columns=cp).round(4))

print("Loadings (autovetor x raiz do autovalor):")
display(pd.DataFrame(pca.components_.T * np.sqrt(pca.explained_variance_), index=ATRIBUTOS, columns=cp).round(4))

trust["PCA"] = trustworthiness(X, Z_pca[:, :2], n_neighbors=10)
scatter2d(Z_pca, "PCA - Iris padronizada",
          f"PC1 ({pca.explained_variance_ratio_[0]:.2%})", f"PC2 ({pca.explained_variance_ratio_[1]:.2%})")

### 2 Kernel PCA (RBF)

In [ ]:
kpca = KernelPCA(n_components=2, kernel="rbf", gamma=GAMMA_KERNEL, eigen_solver="dense")
Z_kpca = kpca.fit_transform(X)
print("Autovalores do kernel centralizado:", np.round(kpca.eigenvalues_, 4))
print("Parâmetros:", kpca.get_params())
trust["Kernel PCA"] = trustworthiness(X, Z_kpca, n_neighbors=10)
scatter2d(Z_kpca, "Kernel PCA (RBF) - Iris padronizada")

### 3 t-SNE

In [ ]:
tsne = TSNE(n_components=2, perplexity=PERPLEXIDADE, learning_rate="auto", init="pca",
            max_iter=1000, random_state=SEED_Q2)
Z_tsne = tsne.fit_transform(X)
print("Divergência KL final:", tsne.kl_divergence_)
print("Iterações:", tsne.n_iter_, "| taxa de aprendizado efetiva:", tsne.learning_rate_)
trust["t-SNE"] = trustworthiness(X, Z_tsne, n_neighbors=10)
scatter2d(Z_tsne, "t-SNE - Iris padronizada")

### 4 UMAP

In [ ]:
umap = UMAP(n_components=2, n_neighbors=VIZINHOS_UMAP, min_dist=MIN_DIST_UMAP,
            metric="euclidean", random_state=SEED_Q2, n_jobs=1)
Z_umap = umap.fit_transform(X)
print("Parâmetros:", dict(n_neighbors=VIZINHOS_UMAP, min_dist=MIN_DIST_UMAP, metric="euclidean"))
trust["UMAP"] = trustworthiness(X, Z_umap, n_neighbors=10)
scatter2d(Z_umap, "UMAP - Iris padronizada")

### 5 Comparação das técnicas

In [ ]:
display(pd.Series(trust, name="Trustworthiness (10 vizinhos)").round(4).to_frame())